# SwapeDev: Video Character Inpainting Pipeline on Kaggle GPU
===============================================================

This notebook deploys a headless, automated video-to-video character replacement module hosted on a free Kaggle GPU (T4 / P100, 15GB–16GB VRAM).

### Pipeline Overview:
1. **Video Ingestion:** 24fps normalization, vertical aspect ratio (512x896 / 576x1024), audio isolation.
2. **Auto-Segmentation (SAM 2):** Point/box tracking of the central actor with 3–5px dilation and Gaussian feathering.
3. **ControlNet Preprocessing:** DWPose skeletal extraction and Depth Anything V2 volumetric depth.
4. **Diffusion Pass:** Stable Diffusion 1.5 + AnimateDiff V3 motion module + MultiControlNet + Character LoRA.
5. **Compositing & Assembly:** Alpha-blending of character over the original plate and FFmpeg audio multiplexing.
6. **API & Tunneling:** FastAPI service exposed via Cloudflare Quick Tunnel or pyngrok with SwapeDev handshake.

In [ ]:
# [1] Clone / Pull SwapeDev or navigate to working directory
!mkdir -p /kaggle/working/video_inpainting
%cd /kaggle/working/video_inpainting

In [ ]:
# [2] Run Environment Setup & Weights Caching
# This installs FFmpeg, diffusers, controlnet_aux, SAM 2, and downloads model weights
!bash /kaggle/working/video_inpainting/setup_kaggle.sh

In [ ]:
# [3] Configure Environment & Secrets (Optional)
import os

# If using ngrok, provide token here:
os.environ["NGROK_AUTHTOKEN"] = ""

# If integrating with SwapeDev Orchestrator & Upstash Redis:
os.environ["PROFILE_ID"] = "default_profile"
os.environ["SWAPEDEV_ORCHESTRATOR_URL"] = ""
os.environ["SWAPEDEV_WEBHOOK_TOKEN"] = "swapedev_secure_worker_secret_2026"
os.environ["UPSTASH_REDIS_REST_URL"] = ""
os.environ["UPSTASH_REDIS_REST_TOKEN"] = ""

print("Environment variables configured successfully.")

In [ ]:
# [4] Start Video Inpainting FastAPI Server with Reverse Tunnel
!python3 /kaggle/working/video_inpainting/server.py

In [ ]:
# [5] Local Test Client: Inpaint a Sample Video Directly
import requests

payload = {
    "video_url": "https://github.com/intel-iot-devkit/sample-videos/raw/master/face-demographics-walking.mp4",
    "character_lora": "spiderman",
    "prompt": "spiderman suit, highly detailed, cinematic lighting, marvel superhero, 8k",
    "negative_prompt": "deformed, blurry, bad anatomy, human face, glitch, boiling artifacts",
    "denoise": 0.8,
    "num_inference_steps": 25,
    "guidance_scale": 7.0,
    "lora_weight": 0.85,
}

response = requests.post("http://127.0.0.1:8189/process-shot", json=payload)
print("Response:", response.status_code, response.json())